# Benchmarking sparse (CSR) vs dense vs LoRA adapter performance

### Load SpaRTA adapter from a checkpoint

In [1]:
from vllm.lora.sparse_adapter.sparse_adapter import SparseAdapter
import torch

In [2]:
sparse_adapter_dir = "/dccstor/disk01/peft_models/gemma-2b-it_sparse_sst2/"

In [3]:
sparse_adapter = SparseAdapter.from_local_checkpoint(
    sparse_adapter_dpath=sparse_adapter_dir,
    sparse_adapter_id=1,
    device="cuda",
)

/u/jriosal/vllm_sparta/my_vllm/vllm/lora/sparse_adapter/sparse_adapter.py:67: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  sparse_delta = torch.sparse_csr_tensor(crow_indices, col_indices, values, size=size)


Loaded sparse adapter from /dccstor/disk01/peft_models/gemma-2b-it_sparse_sst2/


### Inspect adapter

In [4]:
print(f"{'Layer Name':36}{'Layout':15}{'Shape':17}{'NNZ':6}{'Density'}")
print("-" * 81)

for layer_name, delta in sparse_adapter.deltas.items():
    if delta.layout == torch.sparse_csr:
        nnz = delta.values().numel()
    elif delta.layout == torch.strided:
        nnz = delta.numel()
    else:
        raise ValueError(f"unsupported layout: {delta.layout}")
    density = nnz / delta.numel() * 100
    print(f"{layer_name:36}{str(delta.layout)[6:]:15}"
          f"{str(list(delta.shape)):14}{nnz:8,d}{density:7.0f}%")
    

Layer Name                          Layout         Shape            NNZ   Density
---------------------------------------------------------------------------------
model.layers.1.self_attn.o_proj     sparse_csr     [2048, 2048]    49,480      1%
model.layers.11.self_attn.o_proj    sparse_csr     [2048, 2048]    49,397      1%
model.layers.2.self_attn.o_proj     sparse_csr     [2048, 2048]    49,439      1%
model.layers.0.self_attn.v_proj     sparse_csr     [256, 2048]      6,265      1%
model.layers.14.self_attn.o_proj    sparse_csr     [2048, 2048]    49,128      1%
model.layers.4.self_attn.v_proj     sparse_csr     [256, 2048]      6,182      1%
model.layers.7.self_attn.o_proj     sparse_csr     [2048, 2048]    48,938      1%
model.layers.17.self_attn.o_proj    sparse_csr     [2048, 2048]    49,792      1%
model.layers.12.self_attn.v_proj    sparse_csr     [256, 2048]      6,115      1%
model.layers.4.self_attn.o_proj     sparse_csr     [2048, 2048]    49,282      1%
model.layers.9.s

### sparse vs dense benchmarking

when to convert a sparse delta into a dense format, trading memory for faster inference on a GPU

select sparse delta

In [5]:
layer_name = 'model.layers.0.self_attn.o_proj'
delta = sparse_adapter.deltas[layer_name]

In [6]:
S = delta.t()
input_dim = S.shape[0]
# S = S.to('cpu')
# S = S.float()
print('device =', S.device)
print('dtype  =', S.dtype)
print('shape  =', list(S.shape))
print('layout =', S.layout)

device = cuda:0
dtype  = torch.bfloat16
shape  = [2048, 2048]
layout = torch.sparse_csc


create a dense version of delta 

In [7]:
D = S.to_dense()

create inputs

In [8]:
v = torch.ones(input_dim, device=S.device, dtype=S.dtype)    
v.shape

torch.Size([2048])

In [9]:
batch_size = 8
x = torch.ones(batch_size, input_dim, device=S.device, dtype=S.dtype)
x.shape

torch.Size([8, 2048])

`torch.mv` vs `torch.matmul`

In [10]:
S_transpose = S.t()
D_transpose = D.t()

In [21]:
%%time
torch.cuda.synchronize()
for _ in range(10000):
    y_spmv = torch.mv(S_transpose, v) # v @ S
torch.cuda.synchronize()

CPU times: user 134 ms, sys: 1.01 ms, total: 135 ms
Wall time: 134 ms


In [29]:
%%time
torch.cuda.synchronize()
for _ in range(10000):
    y_mv = torch.matmul(v, D) # torch.mv(D_transpose, v) # v @ D
torch.cuda.synchronize()

CPU times: user 149 ms, sys: 997 μs, total: 150 ms
Wall time: 150 ms


In [30]:
torch.allclose(y_spmv, y_mv, atol=1e-3) # (y_spmv - y_mv).abs().max()

True

In [43]:
%%time
torch.cuda.synchronize()
for _ in range(10000):
    y_spmm = torch.matmul(x, S) # x @ S
torch.cuda.synchronize()

CPU times: user 337 ms, sys: 127 ms, total: 464 ms
Wall time: 466 ms


In [54]:
%%time
torch.cuda.synchronize()
for _ in range(10000):
    y_mm = torch.matmul(x, D) # x @ D
torch.cuda.synchronize()

CPU times: user 129 ms, sys: 14.6 ms, total: 143 ms
Wall time: 143 ms


In [55]:
torch.allclose(y_spmm, y_mm, atol=1e-3) # (y_spmm - y_mm).abs().max()

True

output dim

In [57]:
S.shape[1]

2048

In [58]:
y_spmv.shape

torch.Size([2048])

In [59]:
y_mv.shape

torch.Size([2048])

In [60]:
y_spmm.shape

torch.Size([8, 2048])

In [61]:
y_mm.shape

torch.Size([8, 2048])

CUDA graphs

In [62]:
sparse = False
batch = True

label = f"{'Sparse' if sparse else 'Dense'}-{'GEMM' if batch else 'MV'}"

M = S if sparse else D

if not batch:
    M_transpose = M.t()

warmup

In [63]:
for _ in range(20):
    y = torch.matmul(x, M) if batch else torch.mv(M_transpose, v)
torch.cuda.synchronize()

capture: record the graph over one iteration

In [64]:
g = torch.cuda.CUDAGraph()
with torch.cuda.graph(g):
    for _ in range(10000):
        y = torch.matmul(x, M) if batch else torch.mv(M_transpose, v)

replay

In [65]:
%%time
torch.cuda.synchronize()
g.replay()
torch.cuda.synchronize()
print(label)

Dense-GEMM
CPU times: user 109 ms, sys: 4.83 ms, total: 114 ms
Wall time: 113 ms


Summary:

 ||GEMM (batch=8)| MV (batch=1)|
 |--------|:------:|:------:|
 |  Dense  | 113 ms | 87 ms |
 | Sparse  | 443 ms | 112 ms|


sparse not better than dense on GPU at 99% sparsity. Check for higher sparsity (less density)

CUDA graph improvements: 
```
Dense-GEMM  143 -> 113 ms
Dense-MV    150 ->  87 ms
Sparse-GEMM 466 -> 443 ms
Sparse-MV   134 -> 112 ms
```


### LoRA
SpaRTA vs LoRA benchmarking: which one is faster during inference

load and inspect a LoRA adapter

In [1]:
from safetensors.torch import load_file
import torch

In [2]:
lora_path = "/dccstor/disk01/peft_models/policy-scenarios/gemma/lora/adapter/"

In [3]:
lora_weights = load_file(f"{lora_path}adapter_model.safetensors", device="cuda")

In [4]:
for name, tensor in lora_weights.items():
      print(f"{name:70}{str(list(tensor.shape))}")  

base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight        [8, 2048]
base_model.model.model.layers.0.self_attn.q_proj.lora_B.weight        [2048, 8]
base_model.model.model.layers.0.self_attn.v_proj.lora_A.weight        [8, 2048]
base_model.model.model.layers.0.self_attn.v_proj.lora_B.weight        [256, 8]
base_model.model.model.layers.1.self_attn.q_proj.lora_A.weight        [8, 2048]
base_model.model.model.layers.1.self_attn.q_proj.lora_B.weight        [2048, 8]
base_model.model.model.layers.1.self_attn.v_proj.lora_A.weight        [8, 2048]
base_model.model.model.layers.1.self_attn.v_proj.lora_B.weight        [256, 8]
base_model.model.model.layers.10.self_attn.q_proj.lora_A.weight       [8, 2048]
base_model.model.model.layers.10.self_attn.q_proj.lora_B.weight       [2048, 8]
base_model.model.model.layers.10.self_attn.v_proj.lora_A.weight       [8, 2048]
base_model.model.model.layers.10.self_attn.v_proj.lora_B.weight       [256, 8]
base_model.model.model.layers.11.self_attn.

In [5]:
layer_name = "base_model.model.model.layers.0.self_attn.q_proj"
A = lora_weights[f"{layer_name}.lora_A.weight"] # shape: [r, in]
B = lora_weights[f"{layer_name}.lora_B.weight"] # shape: [out, r]

In [6]:
A, B = A.t(), B.t()
A, B = A.to(torch.bfloat16), B.to(torch.bfloat16) # 'cpu'

for n, t in zip(['A', 'B'], [A, B]):
    print(f"{n}:")
    print('device =', t.device)
    print('dtype  =', t.dtype)
    print('shape  =', list(t.shape))
    print('\n')
    

A:
device = cuda:0
dtype  = torch.bfloat16
shape  = [2048, 8]


B:
device = cuda:0
dtype  = torch.bfloat16
shape  = [8, 2048]




In [7]:
input_dim = A.shape[0]
input_dim

2048

re-create inputs

In [8]:
v = torch.ones(input_dim, device=A.device, dtype=A.dtype)    
v.shape

torch.Size([2048])

In [9]:
batch_size = 8
x = torch.ones(batch_size, input_dim, device=A.device, dtype=A.dtype)
x.shape

torch.Size([8, 2048])

LoRA benchmarking

In [15]:
%%time
torch.cuda.synchronize()
for _ in range(10000):
    y_lora = (x @ A) @ B
torch.cuda.synchronize()

CPU times: user 232 ms, sys: 1.35 ms, total: 233 ms
Wall time: 234 ms


In [22]:
%%time
torch.cuda.synchronize()
for _ in range(10000):
    y_lora = (v @ A) @ B
torch.cuda.synchronize()

CPU times: user 258 ms, sys: 1.37 ms, total: 260 ms
Wall time: 261 ms


In [24]:
A_transpose, B_transpose = A.t(), B.t()

In [33]:
%%time
torch.cuda.synchronize()
for _ in range(10000):
    y_lora = torch.mv(B_transpose, torch.mv(A_transpose, v))
torch.cuda.synchronize()                     

CPU times: user 202 ms, sys: 1.7 ms, total: 203 ms
Wall time: 203 ms


Summary:

 || GEMM (batch=8)| MV (batch=1)|
 |------------|:------:|:------:|
 | **SpaRTA** | 466 ms | 134 ms |
 |  **LoRA**  | 234 ms | 261 ms |
 